# Health insurance claims on Parity 1.0

This notebook shows the saved results used in the report and slides. The 26 cases and reference answers are from the published [Parity 1.0 task set](https://paritybench.ai/tasks). The 12 benefit cases and two contested claims form the payment comparison. Twelve prior authorization requests form a separate documentation test. No new cases or reference answers were fabricated.

The experiment used Qwen2.5-0.5B-Instruct Q4_K_M with llama.cpp. The same model saw each payment case once in the direct approach and once in the hybrid approach. It also saw each authorization request once. `analyze.py` scored the saved predictions against the published answers. The cells below read the archived scores and can run offline.


In [1]:
import json
from pathlib import Path

results = Path('results')
summary = json.loads((results / 'summary.json').read_text(encoding='utf-8'))
scores = json.loads((results / 'case_scores.json').read_text(encoding='utf-8'))
predictions = [json.loads(line) for line in (results / 'predictions.jsonl').read_text(encoding='utf-8').splitlines()]
assert len(scores) == len(predictions) == 40
assert len({row['id'] for row in scores}) == 26
print('Published cases:', len({row['id'] for row in scores}))
print('Saved inference calls:', len(predictions))
print('Payment cases:', summary['direct']['n'])
print('Authorization cases:', summary['documentation']['n'])


Published cases: 26
Saved inference calls: 40
Payment cases: 14
Authorization cases: 12


## Where the model and calculator fit

The direct model selected the cost sharing rule and returned payment fields. The hybrid model selected a rule and extracted policy and claim values. Python then calculated the payment split from those values. The same original case text went to both routes. A separate calculator control used checked source values to test the arithmetic implementation; it is not model accuracy.

For payment, the strict amount measure requires **both member and plan amounts** to match the published answer. We also score the underlying rule mode and every published field separately. Authorization uses decision and criterion set scores, with its own denominator.


In [2]:
direct, hybrid, pa = (summary[k] for k in ('direct', 'hybrid', 'documentation'))
print('Payment measure'.ljust(38), 'Direct', 'Hybrid')
for label, key in [('Correct cost sharing mode', 'mode'),
                   ('Both payment amounts correct', 'amount'),
                   ('All published fields correct', 'all_fields'),
                   ('Governing document correct (of 2)', 'governing_document'),
                   ('Governing section correct (of 2)', 'governing_section')]:
    n = 2 if 'Governing' in label else 14
    print(label.ljust(38), f"{direct[key]}/{n}".ljust(6), f"{hybrid[key]}/{n}")
print('Authorization decisions:', f"{pa['decision']}/12")
print('Authorization decision and citations:', f"{pa['decision_and_criteria']}/12")
print('Correct holds:', f"{pa['pend_correct']}/{pa['pend_n']}")
print('False holds:', f"{pa['false_pend']}/9")


Payment measure                        Direct Hybrid
Correct cost sharing mode              7/14   7/14
Both payment amounts correct           0/14   4/14
All published fields correct           0/14   1/14
Governing document correct (of 2)      2/2    2/2
Governing section correct (of 2)       1/2    1/2
Authorization decisions: 4/12
Authorization decision and citations: 0/12
Correct holds: 1/3
False holds: 3/9


## The emergency exception: BEN-006 and BEN-007

Both published claims have a $6,400 allowed amount. Admission waives the emergency department copay in BEN-006; the patient owes the remaining $600 deductible plus 20% of $5,800, or $1,760. Discharge in BEN-007 leaves a $400 copay. The published case links are [BEN-006](https://paritybench.ai/items/BEN-006) and [BEN-007](https://paritybench.ai/items/BEN-007).


In [3]:
for case_id in ('BEN-006', 'BEN-007'):
    for method in ('direct', 'hybrid'):
        row = next(x for x in scores if x['id'] == case_id and x['method'] == method)
        print(case_id, method, 'mode:', row['mode'], 'reference:', row['expected_mode'],
              'member:', row['member'], 'reference:', row['expected_member'])
print('Checked BEN-006 arithmetic:', 600 + 0.20 * (6400 - 600))
hybrid_raw = next(x['prediction'] for x in predictions if x['id'] == 'BEN-006' and x['method'] == 'hybrid')
print('Hybrid BEN-006 extracted deductible limit:', hybrid_raw['deductible_limit'])
print('Hybrid BEN-006 extracted member coinsurance percent:', hybrid_raw['coinsurance_percent'])


BEN-006 direct mode: deductible_then_coinsurance reference: deductible_then_coinsurance member: 100 reference: 1760
BEN-006 hybrid mode: deductible_then_coinsurance reference: deductible_then_coinsurance member: 6140.0 reference: 1760
BEN-007 direct mode: deductible_then_coinsurance reference: copay member: 1 reference: 400
BEN-007 hybrid mode: deductible_then_coinsurance reference: copay member: 1760.0 reference: 400
Checked BEN-006 arithmetic: 1760.0
Hybrid BEN-006 extracted deductible limit: 6000.0
Hybrid BEN-006 extracted member coinsurance percent: 80.0


## Conflicting policy documents: ADJ-017 and ADJ-018

The [MRI case](https://paritybench.ai/items/ADJ-017) says the certificate takes precedence over the summary. The published member cost is $940. The [diabetes education case](https://paritybench.ai/items/ADJ-018) says a mandate controls the benefit and the member cost is $0. Both approaches identified the governing document in both cases, while only one governing section was correct. An exact amount can still result from the wrong cost sharing mode.


In [4]:
for case_id in ('ADJ-017', 'ADJ-018'):
    for method in ('direct', 'hybrid'):
        row = next(x for x in scores if x['id'] == case_id and x['method'] == method)
        print(case_id, method, 'member:', row['member'], 'reference:', row['expected_member'],
              'mode correct:', row['mode_ok'], 'both amounts correct:', row['amount_ok'])


ADJ-017 direct member: 1 reference: 940 mode correct: True both amounts correct: False
ADJ-017 hybrid member: 0.0 reference: 940 mode correct: True both amounts correct: False
ADJ-018 direct member: 1 reference: 0 mode correct: False both amounts correct: False
ADJ-018 hybrid member: 0.0 reference: 0 mode correct: False both amounts correct: True


## Missing documentation in prior authorization

Parity labels a request `pend` when the needed documentation has not been supplied. The three published hold cases are [PA-004](https://paritybench.ai/items/PA-004), [PA-010](https://paritybench.ai/items/PA-010), and [PA-022](https://paritybench.ai/items/PA-022). The model denied the first two and held the third without citing its required E3 criterion. It also held three requests that the benchmark did not mark as holds.


In [5]:
for case_id in ('PA-004', 'PA-010', 'PA-022'):
    row = next(x for x in scores if x['id'] == case_id and x['method'] == 'documentation')
    print(case_id, 'reference:', row['expected_decision'], row['expected_criteria'],
          'model:', row['decision'], row['criteria'])


PA-004 reference: pend ['D2'] model: deny ['A1', 'A2', 'A3', 'A4', 'A5', 'A6']
PA-010 reference: pend ['E1'] model: deny ['A', 'B', 'C', 'D', 'E1', 'E2', 'E3']
PA-022 reference: pend ['E3'] model: pend ['A1', 'A2', 'A3', 'A4', 'B1', 'B2']


## Calculator control and limits

The calculator reproduced every published numeric field for all 14 payment cases when given checked source values. This confirms its implementation on this subset. The model still had to identify the applicable rule and extract the right numbers. The study uses one small model, one run per case, and a selected set of synthetic benchmark items. Its rates do not estimate production claim performance.

[FinQA](https://aclanthology.org/2021.emnlp-main.300/) motivated the separation of source values, calculation, and answer. [ConditionalQA](https://aclanthology.org/2022.acl-long.253/) motivated testing conditional rules and missing evidence.


In [6]:
control = json.loads((results / 'calculator_check.json').read_text(encoding='utf-8'))
print('Cases reproduced with checked inputs:', sum(x['numeric_fields_match'] for x in control.values()),
      'of', len(control))
print('Hybrid payment splits correct:', hybrid['amount'], 'of', hybrid['n'])
print('Direct payment splits correct:', direct['amount'], 'of', direct['n'])


Cases reproduced with checked inputs: 14 of 14
Hybrid payment splits correct: 4 of 14
Direct payment splits correct: 0 of 14
